# Day 18 · Colab 1 — Claude Agent + FastAPI Backend + Redis Memory

**Agentic Systems Bootcamp**

In this notebook you build a small but complete **tool-using agent**:

- a **FastAPI** backend exposing an `/orders/{id}` endpoint (your *external API*),
- a **Redis** memory layer (short-term conversation history + long-term facts),
- a **Claude** agent that calls tools in a `while` loop driven by `stop_reason`.

Everything runs **inside Colab** with no external servers:
`fakeredis` stands in for Redis and FastAPI's `TestClient` calls the API in-process.
Swap either for the real thing by changing one line each (shown at the end).

> **Pattern recap (from the deck):** client tools return `stop_reason="tool_use"`; *your* code runs the
> tool and returns a `tool_result` block; you loop until `stop_reason="end_turn"`.

## Step 1 — Install dependencies & set your API key

`fakeredis` gives us a real Redis API surface in-process. `httpx` is needed by FastAPI's test client.

In [20]:
!pip install -q anthropic fastapi 'httpx<0.28' fakeredis 2>/dev/null
print('deps installed')

deps installed


In [ ]:
import os, getpass

# In Colab, prefer the Secrets panel (key icon) named ANTHROPIC_API_KEY.
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass

if not os.environ.get('ANTHROPIC_API_KEY'):
    # Fallback: paste it (hidden). Leave blank to run in OFFLINE mock mode.
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank = offline mock): ')

LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'  # fast + cheap for a workshop; opus-4-8 for harder reasoning
print('LIVE mode' if LIVE else 'OFFLINE mock mode (no key) — agent loop will be simulated')
print(os.environ.get('ANTHROPIC_API_KEY'))

## Step 2 — A tiny FastAPI backend (the 'external API')

This is the kind of service your agent does **not** control — it just calls it.
We expose one route, then wrap the app in a `TestClient` so calls run in-process.

In [22]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

_ORDERS = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',  'total': 129.0},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',          'qty': 2, 'status': 'processing','total': 58.0},
    'A1003': {'id': 'A1003', 'item': '4K monitor',         'qty': 1, 'status': 'delivered', 'total': 410.0},
}

@app.get('/orders/{order_id}')
def get_order(order_id: str):
    o = _ORDERS.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404, detail='order not found')
    return o

client = TestClient(app)
print(client.get('/orders/A1001').json())

{'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped', 'total': 129.0}


## Step 3 — A Redis memory layer

Two responsibilities, the way the deck framed memory:

- **Short-term** — the running conversation, stored as a Redis **list** per session (`hist:<session>`).
- **Long-term** — durable **facts** about the user, stored in a Redis **hash** (`facts:<session>`), with an optional TTL.

`fakeredis` implements the same commands, so this class is unchanged when you move to real Redis.

In [23]:
import json, time, fakeredis

class RedisMemory:
    def __init__(self, r, session_id: str, history_limit: int = 40):
        self.r = r
        self.sid = session_id
        self.history_limit = history_limit
        self.h_key = f'hist:{session_id}'
        self.f_key = f'facts:{session_id}'

    # ---- short-term: conversation turns ----
    def append_turn(self, role: str, content):
        self.r.rpush(self.h_key, json.dumps({'role': role, 'content': content}))
        self.r.ltrim(self.h_key, -self.history_limit, -1)  # keep only the tail

    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]

    # ---- long-term: durable facts ----
    def set_fact(self, key: str, value: str, ttl_seconds: int | None = None):
        self.r.hset(self.f_key, key, value)
        if ttl_seconds:
            self.r.expire(self.f_key, ttl_seconds)

    def get_fact(self, key: str):
        v = self.r.hget(self.f_key, key)
        return v.decode() if isinstance(v, bytes) else v

    def all_facts(self):
        return {k.decode(): v.decode() for k, v in self.r.hgetall(self.f_key).items()}

r = fakeredis.FakeStrictRedis()
mem = RedisMemory(r, session_id='demo-user')
mem.set_fact('name', 'Asha')
mem.append_turn('user', 'hello')
print('facts:', mem.all_facts())
print('history:', mem.load_history())

facts: {'name': 'Asha'}
history: [{'role': 'user', 'content': 'hello'}]


## Step 4 — Declare the tools (JSON schemas)

Three client tools. Names are **namespaced by purpose** and every field is described —
the description *is* the prompt the model reads when deciding how to call.
`order_id` uses a `pattern` so the model returns well-formed IDs.

In [24]:
TOOLS = [
    {
        'name': 'get_order',
        'description': 'Look up a customer order by its ID and return item, quantity, status and total.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'string', 'description': "Order ID like 'A1001'.", 'pattern': '^[Aa][0-9]{4}$'}
            },
            'required': ['order_id'],
        },
    },
    {
        'name': 'remember_fact',
        'description': 'Persist a durable fact about the user (e.g. shipping preference) for future turns.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'key':   {'type': 'string', 'description': 'Short fact key, e.g. "shipping_pref".'},
                'value': {'type': 'string', 'description': 'The fact value to store.'},
            },
            'required': ['key', 'value'],
        },
    },
    {
        'name': 'recall_fact',
        'description': 'Retrieve a previously stored fact about the user by key. Returns empty if unknown.',
        'input_schema': {
            'type': 'object',
            'properties': {'key': {'type': 'string', 'description': 'The fact key to look up.'}},
            'required': ['key'],
        },
    },
]
print(len(TOOLS), 'tools declared')

3 tools declared


## Step 5 — A dispatch map: tool name → Python function

This is the boundary between *the model's intent* and *your code*. Each function returns a
string (what the model will read back). Errors are **returned**, not raised, so the agent can recover —
that maps to `is_error: true` on the `tool_result` block in Step 6.

In [25]:
def tool_get_order(order_id: str):
    resp = client.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    resp.raise_for_status()
    return resp.json()

def tool_remember_fact(key: str, value: str):
    mem.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}

def tool_recall_fact(key: str):
    v = mem.get_fact(key)
    return {'key': key, 'value': v} if v is not None else {'key': key, 'value': None}

DISPATCH = {
    'get_order': tool_get_order,
    'remember_fact': tool_remember_fact,
    'recall_fact': tool_recall_fact,
}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        is_err = isinstance(out, dict) and 'error' in out
        return out, is_err
    except Exception as e:
        return {'error': repr(e)}, True

print(run_tool('get_order', {'order_id': 'A1002'}))
print(run_tool('get_order', {'order_id': 'A9999'}))

({'id': 'A1002', 'item': 'USB-C hub', 'qty': 2, 'status': 'processing', 'total': 58.0}, False)
({'error': 'No order A9999 found.'}, True)


## Step 6 — The agent loop (driven by `stop_reason`)

This is the heart of client-side tool use:

1. send messages + tool definitions,
2. if `stop_reason == 'tool_use'`: run **every** `tool_use` block, append a single user message
   containing one `tool_result` per call (matched by `tool_use_id`), and loop,
3. stop at `stop_reason == 'end_turn'` and return the text.

If there's no API key we simulate one scripted tool call so the notebook still runs end-to-end.

In [38]:
import json

SYSTEM = (
    'You are an order-support assistant. Use get_order for any order question. '
    'Use remember_fact / recall_fact to keep durable user preferences across turns. '
    'Be concise.'
)

def agent_turn(user_text, max_steps=6, verbose=True):
    mem.append_turn('user', user_text)
    messages = mem.load_history()

    if not LIVE:
        # ---- offline mock: pretend the model asked for get_order once ----
        if verbose: print('… (mock) model requests get_order A1001')
        out, _ = run_tool('get_order', {'order_id': 'A1001'})
        reply = f'(mock) Order A1001 is {out.get("status", "?")}.'
        mem.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages,
        )
        print_token_usage(resp)
        if resp.stop_reason == 'tool_use':
            # echo the assistant turn (text + tool_use blocks) back into the transcript
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose: print(f'  → tool: {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            continue
        # end_turn
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem.append_turn('assistant', text)
        return text
    return '(stopped: max steps reached)'

print(agent_turn('What is the status of order A1002?'))

📊 Tokens this step: in=1577, out=66
📦 Cumulative tokens: in=1577, out=66
  → tool: get_order({'order_id': 'A1002'})
📊 Tokens this step: in=1702, out=99
📦 Cumulative tokens: in=3279, out=165
Here's the status for order **A1002**:

| Field | Info |
|-------|------|
| **Item** | USB-C Hub |
| **Quantity** | 2 |
| **Status** | 🔄 Processing |
| **Total** | $58.00 |

The order is currently being **processed** and hasn't shipped yet. Is there anything else you'd like to know?


## Step 7 — Persistence check: memory survives across turns

Because every turn is written to Redis, a fresh `agent_turn` call still sees the history and facts.
Ask the agent to remember something, then recall it in a later turn.

In [27]:
print(agent_turn('Please remember that my shipping preference is express.'))
print('---')
print(agent_turn('What did I say my shipping preference was?'))
print('---')
print('Raw facts in Redis:', mem.all_facts())
print('History length:', len(mem.load_history()), 'turns')

  → tool: remember_fact({'key': 'shipping_pref', 'value': 'express'})
Got it! I've saved your shipping preference as **express**. I'll keep that in mind for future interactions. Is there anything else I can help you with?
---
  → tool: recall_fact({'key': 'shipping_pref'})
Your shipping preference is set to **express**. Is there anything else I can help you with?
---
Raw facts in Redis: {'name': 'Asha', 'shipping_pref': 'express'}
History length: 7 turns


## Step 8 — Multi-turn chat demo

A short scripted conversation that exercises lookup + memory together.

In [28]:
for msg in [
    'Hi, I am Asha.',
    'How much was order A1003?',
    'Remember that my budget cap is 500 dollars.',
    'Given my budget cap, was that order within it?',
]:
    print('USER:', msg)
    print('AGENT:', agent_turn(msg, verbose=False))
    print()

USER: Hi, I am Asha.
AGENT: Got it, Asha! How can I assist you today?

USER: How much was order A1003?
AGENT: Order **A1003** was for a **4K Monitor** (qty: 1) and the total was **$410.00**. It has been **delivered**. Is there anything else I can help you with, Asha?

USER: Remember that my budget cap is 500 dollars.
AGENT: Done! I've saved your budget cap as **$500**. Is there anything else I can help you with, Asha?

USER: Given my budget cap, was that order within it?
AGENT: Yes! Order A1003 totaled **$410.00**, which is within your **$500 budget cap**. Is there anything else I can help you with, Asha?



## Extension tasks

Pick a few — these are the assignment for this lab.

1. **Rolling summary / compaction.** When `load_history()` exceeds N turns, summarise the oldest
   turns with a cheap model call and replace them with one synthetic `assistant` summary message.
   Keep total context bounded.
2. **TTL & a `forget` tool.** Add a `forget_fact(key)` tool and give facts a TTL via `set_fact(..., ttl_seconds=...)`.
   Show that an expired fact returns `None`.
3. **Parallel lookups.** Add a `get_customer` tool and a second order endpoint, then ask a question that
   needs both — observe Claude emit **two `tool_use` blocks in one turn**. Confirm your loop answers both ids.
4. **Guarded writes / PII.** Reject `remember_fact` values that look like card numbers or emails (regex);
   return `is_error: true` with a helpful message and confirm the agent recovers.
5. **Token accounting.** Read `resp.usage` each step; print cumulative input/output tokens for a turn.
6. **Real Redis.** Replace `fakeredis.FakeStrictRedis()` with `redis.Redis.from_url(os.environ['REDIS_URL'])`
   (e.g. a free Upstash/Redis Cloud URL). The `RedisMemory` class should not change at all.

> **Stretch:** swap `TestClient` for a real deployed FastAPI URL using `httpx.Client(base_url=...)`.
> Only `tool_get_order` changes; the agent loop is untouched.

In [29]:

# 1. Rolling summary / compaction

import json

# ---- add summary config to existing mem object ----
if not hasattr(mem, "summary_key"):
    mem.summary_key = f"summary:{mem.sid}"

mem.compact_after_turns = 12
mem.compact_keep_latest = 8

def raw_history(self):
    return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]

def get_summary(self):
    s = self.r.get(self.summary_key)
    if s is None:
        return None
    return s.decode() if isinstance(s, bytes) else s

def set_summary(self, text: str):
    self.r.set(self.summary_key, text)

def cheap_summarise_messages(self, messages):
    # cheap summariser = simple string compression of old turns
    parts = []
    for m in messages:
        role = m.get("role", "unknown")
        content = m.get("content", "")
        if isinstance(content, list):
            content = json.dumps(content)[:200]
        else:
            content = str(content)[:200]
        parts.append(f"{role}: {content}")
    return " | ".join(parts)

def compact_history_if_needed(self, verbose: bool = False):
    hist = self.raw_history()
    if len(hist) <= self.compact_after_turns:
        return

    oldest = hist[:-self.compact_keep_latest]
    latest = hist[-self.compact_keep_latest:]

    old_summary = self.get_summary()
    new_summary = self.cheap_summarise_messages(oldest)

    merged = new_summary if not old_summary else f"{old_summary} | {new_summary}"
    self.set_summary(merged)

    # replace old history with only latest few turns
    self.r.delete(self.h_key)
    for turn in latest:
        self.r.rpush(self.h_key, json.dumps(turn))

    if verbose:
        print(" History compacted into rolling summary.")

def load_history_with_summary(self):
    hist = [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]
    summary = self.get_summary()
    if summary:
        return [
            {
                "role": "assistant",
                "content": f"[Conversation summary of earlier turns]\n{summary}"
            }
        ] + hist
    return hist

# monkey-patch
RedisMemory.raw_history = raw_history
RedisMemory.get_summary = get_summary
RedisMemory.set_summary = set_summary
RedisMemory.cheap_summarise_messages = cheap_summarise_messages
RedisMemory.compact_history_if_needed = compact_history_if_needed
RedisMemory.load_history = load_history_with_summary

print("Extension 1 loaded")

Extension 1 loaded


In [30]:
# TEST — Extension 1
# Reset history for a clean demo
mem.r.delete(mem.h_key)
if hasattr(mem, "summary_key"):
    mem.r.delete(mem.summary_key)

for i in range(15):
    mem.append_turn("user", f"user message {i}")
    mem.append_turn("assistant", f"assistant reply {i}")

print("Before compaction:", len(mem.raw_history()))
mem.compact_history_if_needed(verbose=True)
print("After compaction:", len(mem.raw_history()))
print("\nSummary:\n", mem.get_summary())
print("\nFirst item from load_history():")
print(mem.load_history()[0])

Before compaction: 30
 History compacted into rolling summary.
After compaction: 8

Summary:
 user: user message 0 | assistant: assistant reply 0 | user: user message 1 | assistant: assistant reply 1 | user: user message 2 | assistant: assistant reply 2 | user: user message 3 | assistant: assistant reply 3 | user: user message 4 | assistant: assistant reply 4 | user: user message 5 | assistant: assistant reply 5 | user: user message 6 | assistant: assistant reply 6 | user: user message 7 | assistant: assistant reply 7 | user: user message 8 | assistant: assistant reply 8 | user: user message 9 | assistant: assistant reply 9 | user: user message 10 | assistant: assistant reply 10

First item from load_history():
{'role': 'assistant', 'content': '[Conversation summary of earlier turns]\nuser: user message 0 | assistant: assistant reply 0 | user: user message 1 | assistant: assistant reply 1 | user: user message 2 | assistant: assistant reply 2 | user: user message 3 | assistant: assistan

In [31]:

# TASK 2 — TTL + forget_fact (FIXED VERSION)

def _fact_key(self, key: str) -> str:
    return f"fact:{self.sid}:{key}"

def _fact_index_key(self) -> str:
    return f"facts_index:{self.sid}"

def set_fact_with_ttl(self, key: str, value: str, ttl_seconds: int | None = None):
    redis_key = self._fact_key(key)
    self.r.set(redis_key, value)

    # keep a separate set of known fact keys
    self.r.sadd(self._fact_index_key(), key)

    if ttl_seconds is not None:
        self.r.expire(redis_key, ttl_seconds)

def get_fact_with_ttl(self, key: str):
    v = self.r.get(self._fact_key(key))
    if v is None:
        return None
    return v.decode() if isinstance(v, bytes) else v

def forget_fact(self, key: str):
    deleted = self.r.delete(self._fact_key(key))
    self.r.srem(self._fact_index_key(), key)
    return bool(deleted)

def all_facts_with_ttl(self):
    out = {}
    for k in self.r.smembers(self._fact_index_key()):
        key = k.decode() if isinstance(k, bytes) else k
        v = self.get_fact(key)
        if v is None:
            # clean expired facts out of the index
            self.r.srem(self._fact_index_key(), key)
        else:
            out[key] = v
    return out

# patch RedisMemory methods
RedisMemory._fact_key = _fact_key
RedisMemory._fact_index_key = _fact_index_key
RedisMemory.set_fact = set_fact_with_ttl
RedisMemory.get_fact = get_fact_with_ttl
RedisMemory.forget_fact = forget_fact
RedisMemory.all_facts = all_facts_with_ttl

# ---- add / update tool schemas ----
tool_names = [t["name"] for t in TOOLS]

if "forget_fact" not in tool_names:
    TOOLS.append({
        "name": "forget_fact",
        "description": "Delete a previously stored fact by key.",
        "input_schema": {
            "type": "object",
            "properties": {
                "key": {"type": "string", "description": "Fact key to delete"}
            },
            "required": ["key"],
        },
    })

for t in TOOLS:
    if t["name"] == "remember_fact":
        t["description"] = "Persist a fact about the user, optionally with ttl_seconds."
        t["input_schema"]["properties"]["ttl_seconds"] = {
            "type": "integer",
            "description": "Optional TTL in seconds"
        }

# ---- tool functions ----
def tool_remember_fact(key: str, value: str, ttl_seconds: int | None = None):
    mem.set_fact(key, value, ttl_seconds=ttl_seconds)
    return {"ok": True, "stored": {key: value}, "ttl_seconds": ttl_seconds}

def tool_forget_fact(key: str):
    deleted = mem.forget_fact(key)
    return {"ok": True, "key": key, "deleted": deleted}

DISPATCH["remember_fact"] = tool_remember_fact
DISPATCH["forget_fact"] = tool_forget_fact

print("Extension 2 loaded ")

Extension 2 loaded 


In [32]:
# TEST — Extension 2
import time

# TTL test
mem.set_fact("promo_code", "SAVE20", ttl_seconds=2)
print("Immediately:", mem.get_fact("promo_code"))
time.sleep(3)
print("After expiry:", mem.get_fact("promo_code"))

# forget_fact test
mem.set_fact("color", "blue")
print("Before forget:", mem.get_fact("color"))
print("Forget result:", mem.forget_fact("color"))
print("After forget:", mem.get_fact("color"))

Immediately: SAVE20
After expiry: None
Before forget: blue
Forget result: True
After forget: None


In [33]:
# TASK 3 — Parallel lookups

from fastapi import HTTPException

# Add supporting data
if "_CUSTOMERS" not in globals():
    _CUSTOMERS = {
        "C2001": {"id": "C2001", "name": "Asha Rao", "tier": "Gold", "city": "Bengaluru"},
        "C2002": {"id": "C2002", "name": "Rahul Shah", "tier": "Silver", "city": "Pune"},
    }

if "_ORDER_SUMMARIES" not in globals():
    _ORDER_SUMMARIES = {
        "A1001": {"id": "A1001", "shipping_eta": "2026-06-25", "warehouse": "Bengaluru DC"},
        "A1002": {"id": "A1002", "shipping_eta": "2026-06-27", "warehouse": "Mumbai DC"},
        "A1003": {"id": "A1003", "shipping_eta": "Delivered", "warehouse": "Chennai DC"},
    }

# Ensure customer_id exists on orders
for oid, cid in [("A1001", "C2001"), ("A1002", "C2002"), ("A1003", "C2001")]:
    if oid in _ORDERS and "customer_id" not in _ORDERS[oid]:
        _ORDERS[oid]["customer_id"] = cid

# Add FastAPI routes if not already present
existing_paths = {route.path for route in app.routes}

if "/orders/{order_id}/summary" not in existing_paths:
    @app.get("/orders/{order_id}/summary")
    def get_order_summary(order_id: str):
        s = _ORDER_SUMMARIES.get(order_id.upper())
        if not s:
            raise HTTPException(status_code=404, detail="order summary not found")
        return s

if "/customers/{customer_id}" not in existing_paths:
    @app.get("/customers/{customer_id}")
    def get_customer(customer_id: str):
        c = _CUSTOMERS.get(customer_id.upper())
        if not c:
            raise HTTPException(status_code=404, detail="customer not found")
        return c

# Tool functions
def tool_get_order_summary(order_id: str):
    resp = client.get(f"/orders/{order_id}/summary")
    if resp.status_code == 404:
        return {"error": f"No summary for order {order_id} found."}
    resp.raise_for_status()
    return resp.json()

def tool_get_customer(customer_id: str):
    resp = client.get(f"/customers/{customer_id}")
    if resp.status_code == 404:
        return {"error": f"No customer {customer_id} found."}
    resp.raise_for_status()
    return resp.json()

# Tool schemas
tool_names = [t["name"] for t in TOOLS]

if "get_order_summary" not in tool_names:
    TOOLS.append({
        "name": "get_order_summary",
        "description": "Look up shipping ETA and warehouse details for an order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID like A1001"}
            },
            "required": ["order_id"],
        },
    })

if "get_customer" not in tool_names:
    TOOLS.append({
        "name": "get_customer",
        "description": "Look up a customer by customer_id.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Customer ID like C2001"}
            },
            "required": ["customer_id"],
        },
    })

DISPATCH["get_order_summary"] = tool_get_order_summary
DISPATCH["get_customer"] = tool_get_customer

SYSTEM = (
    "You are an order-support assistant. "
    "Use get_order for order questions. "
    "Use get_order_summary for shipping ETA / warehouse questions. "
    "Use get_customer for customer profile questions. "
    "Use remember_fact / recall_fact / forget_fact to store user preferences. "
    "If multiple independent lookups are needed, you may call multiple tools in one turn. "
    "Be concise."
)

print("Extension 3 loaded")

Extension 3 loaded


In [34]:
# TEST — Extension 3

# 1) Direct tool test to prove the new tools work
print("Order summary:", DISPATCH["get_order_summary"]("A1001"))
print("Customer:", DISPATCH["get_customer"]("C2001"))

# 2) Claude / agent test
# Ask a question that should need order + order summary + customer
agent_turn("For order A1001, tell me the order status, shipping ETA, warehouse, and customer details.")

Order summary: {'id': 'A1001', 'shipping_eta': '2026-06-25', 'warehouse': 'Bengaluru DC'}
Customer: {'id': 'C2001', 'name': 'Asha Rao', 'tier': 'Gold', 'city': 'Bengaluru'}
  → tool: get_order({'order_id': 'A1001'})
  → tool: get_order_summary({'order_id': 'A1001'})
  → tool: get_customer({'customer_id': 'C2001'})


"Here's a full summary for order **A1001**:\n\n---\n\n### 📦 Order Details\n| Field | Info |\n|-------|------|\n| **Item** | Mechanical Keyboard |\n| **Quantity** | 1 |\n| **Status** | ✅ Shipped |\n| **Total** | $129.00 |\n\n---\n\n### 🚚 Shipping Info\n| Field | Info |\n|-------|------|\n| **Shipping ETA** | June 25, 2026 |\n| **Warehouse** | Bengaluru DC |\n\n---\n\n### 👤 Customer Details\n| Field | Info |\n|-------|------|\n| **Name** | Asha Rao |\n| **Customer ID** | C2001 |\n| **Tier** | 🥇 Gold |\n| **City** | Bengaluru |\n\n---\n\nEverything looks on track! Is there anything else you'd like to know?"

In [35]:
# TASK 4 — Guarded writes / PII rejection

import re

EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
CARD_RE  = re.compile(r"\b(?:\d[ -]*?){13,19}\b")

def looks_like_pii(value: str):
    if EMAIL_RE.search(value):
        return "Looks like an email address."
    if CARD_RE.search(value):
        return "Looks like a payment card number."
    return None

def guarded_tool_remember_fact(key: str, value: str, ttl_seconds: int | None = None):
    pii_reason = looks_like_pii(value)
    if pii_reason:
        return {
            "error": f"Refusing to store sensitive data in memory. {pii_reason}",
            "is_error": True,
        }

    mem.set_fact(key, value, ttl_seconds=ttl_seconds)
    return {"ok": True, "stored": {key: value}, "ttl_seconds": ttl_seconds}

DISPATCH["remember_fact"] = guarded_tool_remember_fact

print("Extension 4 loaded")

Extension 4 loaded


In [36]:
# TEST — Extension 4

print(DISPATCH["remember_fact"]("email", "alice@example.com"))
print(DISPATCH["remember_fact"]("card", "4111 1111 1111 1111"))
print(DISPATCH["remember_fact"]("pet_name", "Milo"))

{'error': 'Refusing to store sensitive data in memory. Looks like an email address.', 'is_error': True}
{'error': 'Refusing to store sensitive data in memory. Looks like a payment card number.', 'is_error': True}
{'ok': True, 'stored': {'pet_name': 'Milo'}, 'ttl_seconds': None}


In [37]:
# TASK 5 — Token accounting helper

cumulative_input_tokens = 0
cumulative_output_tokens = 0

def print_token_usage(resp):
    global cumulative_input_tokens, cumulative_output_tokens

    usage = getattr(resp, "usage", None)
    if usage is None:
        return

    in_tok = getattr(usage, "input_tokens", 0) or 0
    out_tok = getattr(usage, "output_tokens", 0) or 0

    cumulative_input_tokens += in_tok
    cumulative_output_tokens += out_tok

    print(f"📊 Tokens this step: in={in_tok}, out={out_tok}")
    print(f"📦 Cumulative tokens: in={cumulative_input_tokens}, out={cumulative_output_tokens}")

print("Extension 5 helper loaded")

Extension 5 helper loaded


In [39]:
print(agent_turn('What is the status of order A1002?'))

📊 Tokens this step: in=1689, out=74
📦 Cumulative tokens: in=4968, out=239
  → tool: get_order({'order_id': 'A1002'})
📊 Tokens this step: in=1822, out=105
📦 Cumulative tokens: in=6790, out=344
Here's the current status for order **A1002**:

| Field | Info |
|-------|------|
| **Item** | USB-C Hub |
| **Quantity** | 2 |
| **Status** | 🔄 Processing |
| **Total** | $58.00 |

The order is still being **processed** and hasn't shipped yet. Would you like more details, such as shipping ETA or customer info?
